# Travel Style Classifier Training

This notebook trains a machine learning model to classify destinations by travel style.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
import joblib
import numpy as np

In [4]:
# Create sample dataset (in real implementation, collect real data)
data = {
    'destination': [
        'Paris, France',
        'Tokyo, Japan',
        'Bali, Indonesia',
        'New York City, USA',
        'Swiss Alps, Switzerland',
        'Dubai, UAE',
        'Machu Picchu, Peru',
        'Santorini, Greece',
        'Iceland',
        'Costa Rica',
        'Barcelona, Spain',
        'Amsterdam, Netherlands',
        'Queenstown, New Zealand',
        'Maldives',
        'Berlin, Germany',
        'Rio de Janeiro, Brazil',
        'Vancouver, Canada',
        'Prague, Czech Republic',
        'Cape Town, South Africa',
        'Hanoi, Vietnam',
        'Beirut, Lebanon'
    ],
    'style': [
        'Culture',
        'Culture',
        'Relaxation',
        'Culture',
        'Adventure',
        'Luxury',
        'Adventure',
        'Relaxation',
        'Adventure',
        'Adventure',
        'Culture',
        'Culture',
        'Adventure',
        'Luxury',
        'Culture',
        'Culture',
        'Adventure',
        'Culture',
        'Adventure',
        'Budget',
        'Budget'
    ]
}

df = pd.DataFrame(data)
print(df.head())
print(df['style'].value_counts())

               destination       style
0            Paris, France     Culture
1             Tokyo, Japan     Culture
2          Bali, Indonesia  Relaxation
3       New York City, USA     Culture
4  Swiss Alps, Switzerland   Adventure
style
Culture       8
Adventure     7
Relaxation    2
Luxury        2
Budget        2
Name: count, dtype: int64


In [5]:
# Split data
X = df['destination']
y = df['style']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

Training set size: 16
Test set size: 5


In [6]:
# Define classifiers to test
classifiers = {
    'RandomForest': RandomForestClassifier(random_state=42),
    'LogisticRegression': LogisticRegression(random_state=42, max_iter=1000),
    'SVM': SVC(random_state=42)
}

# Test each classifier
results = {}
for name, clf in classifiers.items():
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=1000)),
        ('clf', clf)
    ])
    
    # Cross-validation
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')
    
    # Train and test
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    results[name] = {
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'test_accuracy': accuracy_score(y_test, y_pred),
        'classification_report': classification_report(y_test, y_pred, output_dict=True)
    }
    
    print(f"{name}:")
    print(f"  CV Accuracy: {cv_scores.mean():.3f} (+/- {cv_scores.std() * 2:.3f})")
    print(f"  Test Accuracy: {accuracy_score(y_test, y_pred):.3f}")
    print()

/home/hosen/Desktop/Desktop2/AIE/smart-travel-planner/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


RandomForest:
  CV Accuracy: 0.367 (+/- 0.133)
  Test Accuracy: 0.400

LogisticRegression:
  CV Accuracy: 0.317 (+/- 0.067)
  Test Accuracy: 0.400

SVM:
  CV Accuracy: 0.367 (+/- 0.133)
  Test Accuracy: 0.400



/home/hosen/Desktop/Desktop2/AIE/smart-travel-planner/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/hosen/Desktop/Desktop2/AIE/smart-travel-planner/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/hosen/Desktop/Desktop2/AIE/smart-travel-planner/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` par

In [7]:
# Choose best model (RandomForest for this example)
best_model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=1000)),
    ('clf', RandomForestClassifier(random_state=42))
])

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print("Final Model Performance:")
print(classification_report(y_test, y_pred))

Final Model Performance:
              precision    recall  f1-score   support

   Adventure       0.00      0.00      0.00         2
     Culture       0.40      1.00      0.57         2
  Relaxation       0.00      0.00      0.00         1

    accuracy                           0.40         5
   macro avg       0.13      0.33      0.19         5
weighted avg       0.16      0.40      0.23         5



/home/hosen/Desktop/Desktop2/AIE/smart-travel-planner/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/hosen/Desktop/Desktop2/AIE/smart-travel-planner/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/hosen/Desktop/Desktop2/AIE/smart-travel-planner/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` par

In [9]:
%pwd

'/home/hosen/Desktop/Desktop2/AIE/smart-travel-planner/notebooks'

In [11]:
import os
print(f"Current Directory: {os.getcwd()}")
print(f"Files in parent directory: {os.listdir('..')}")

Current Directory: /home/hosen/Desktop/Desktop2/AIE/smart-travel-planner/notebooks
Files in parent directory: ['app', '.venv', 'docker-compose.yml', 'pyproject (Copy).toml', '.git', 'uv.lock', 'main.py', '.pytest_cache', '.gitignore', 'tests', 'pyproject.toml', '.python-version', 'alembic.ini', 'notebooks', 'models', 'frontend', 'Dockerfile', 'alembic', '.env', '.env.example', 'README.md', 'scripts']


In [13]:
# Save the model
joblib.dump(best_model, '../models/classifier.joblib')
print("Model saved to ../models/classifier.joblib")

# Save results
results_df = pd.DataFrame([
    {
        'model': name,
        'cv_accuracy_mean': res['cv_mean'],
        'cv_accuracy_std': res['cv_std'],
        'test_accuracy': res['test_accuracy'],
        'timestamp': pd.Timestamp.now()
    }
    for name, res in results.items()
])

results_df.to_csv('../data/experiment_results.csv', index=False)
print("Results saved to ../data/experiment_results.csv")

Model saved to ../models/classifier.joblib
Results saved to ../data/experiment_results.csv
